[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/bonds_stocks.ipynb)

# Bonds and stocks full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (bond prices, yields to maturity, duration, spot rates, a bond that may default, shares
valued by their dividends, a business valued by its free cash flow), **tier 3** stretch problems (combine ideas,
spot the trap, run a formula backwards) and **tier 4**, the case: Bangalore Cold Chain (a fictional company; our own
data) replaces its bank loans with a ten-year bond, and has to decide what coupon the bond must carry and what its
shares are worth. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same prices, yields and durations in plain
Python, with `numpy` for arrays of payments and `pandas` for a whole list of bonds and for the case's dated
payments, so you can run them on your own numbers. And two cells you can move yourself: a yield that reprices a
short bond and a long one, and a return on new investment that decides whether growth adds value to a share.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, move it and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Money in
dollars, to the cent: `1035.46`. Rates as percentages: `7.00` for 7 % (typing the % sign is fine, and so is the
rate itself, `0.07`). Durations in years, to two decimals: `3.68`. The case's totals in $ millions, to two decimals:
`76.91`. Numbers without thousands separators: `1035.46`, not `1,035.46`. You can type the arithmetic too:
`60/1.05 + 1060/1.05**2`. For several answers, separate them with commas, in the order the question lists them (the
box's label shows the order). You can also put your answer between the quotes in a cell's last line,
`practice("P1.3", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[bonds and stocks cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/bonds_stocks_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # cash in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# cash as it is paid CHALK · money moved to today SKY · the rate VIOLET · growth MINT · a shortcut that misleads CORAL ·
# the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own example · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…), exp(…), ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(dollars?|years?|points?|millions?|percent)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol, pct=False):
    """Within the tolerance; a rate asked for in percent may also be typed as the rate itself (0.07 for 7 %)."""
    if isinstance(got, str): return False
    tol = max(tol, 1e-7)
    return abs(got - want) <= tol or (pct and abs(got * 100 - want) <= tol)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Bond prices and yields',
     "q": 'What is $1,000 received in 5 years worth today at 4 %?',
     "parts": ['present value ($)'],
     "answer": [821.9271067593517], "tol": [0.005], "pct": [], "function": 'PV',
     "working": '1,000/1.04⁵ = 1,000/1.216653 = 821.93'},
    {"id": 'P1.2', "tier": 1, "group": 'Bond prices and yields',
     "q": 'What is the 10-year annuity factor at 6 %? Give four decimals.',
     "parts": ['annuity factor (four decimals)'],
     "answer": [7.360087051414702], "tol": [5e-05], "pct": [], "function": 'PV',
     "working": '1/0.06 − 1/(0.06 × 1.06¹⁰) = 16.6667 − 9.3066 = 7.3601'},
    {"id": 'P1.3', "tier": 1, "group": 'Bond prices and yields',
     "q": 'A 3-year bond has a face value of $1,000 and a 7 % coupon paid once a year. Similar bonds yield 5 %. What is its price?',
     "parts": ['price ($)'],
     "answer": [1054.4649605874097], "tol": [0.005], "pct": [], "function": 'PV',
     "working": '70 × 2.7232 + 1,000/1.05³ = 190.63 + 863.84 = 1,054.46. =-PV(5%, 3, 70, 1000, 0)'},
    {"id": 'P1.4', "tier": 1, "group": 'Bond prices and yields',
     "q": 'A bond pays a $60 coupon a year and sells for $950. What is its current yield, in percent?',
     "parts": ['current yield (%)'],
     "answer": [6.315789473684211], "tol": [0.005], "pct": [0], "function": '—',
     "working": '60/950 = 6.32 % (the coupon over the price; it leaves out the gain as the price climbs back to face value)'},
    {"id": 'P1.5', "tier": 1, "group": 'Duration',
     "q": 'A zero-coupon bond pays $1,000 in 8 years and nothing before. At a yield of 4.5 %, what is its price, and what is its duration in years?',
     "parts": ['price ($)', 'duration (years)'],
     "answer": [703.1851269688576, 8.0], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": '1,000/1.045⁸ = 703.19; one payment, so the average wait is the maturity: 8 years'},
    {"id": 'P1.6', "tier": 1, "group": 'Real rates and default risk',
     "q": 'A bond yields 6 % and prices rise 2 % a year. What is the real rate, in percent (exact, not the shortcut)?',
     "parts": ['real rate (%)'],
     "answer": [3.9215686274509887], "tol": [0.005], "pct": [0], "function": '—',
     "working": '1.06/1.02 − 1 = 3.9216 % (the shortcut 6 − 2 = 4 % is a little high)'},
    {"id": 'P1.7', "tier": 1, "group": 'Stock prices and dividends',
     "q": 'A share sells for $40. It is expected to pay a $1.60 dividend next year and then sell for $42.40. What return do investors expect, in percent?',
     "parts": ['expected return (%)'],
     "answer": [10.0], "tol": [0.005], "pct": [0], "function": '—',
     "working": '(1.60 + 42.40 − 40)/40 = 4/40 = 10 %: a 4 % dividend yield plus a 6 % capital gain'},
    {"id": 'P1.8', "tier": 1, "group": 'Stock prices and dividends',
     "q": 'A share will pay $1.50 next year, growing 4 % a year forever. If shareholders want 9 %, what is it worth?',
     "parts": ['value of the share ($)'],
     "answer": [30.000000000000004], "tol": [0.005], "pct": [], "function": '—',
     "working": 'DIV₁/(r − g) = 1.50/0.05 = 30'},
    {"id": 'P1.9', "tier": 1, "group": 'Growth and the value of a business',
     "q": 'A company plows back 45 % of its earnings and earns 12 % on new investment. How fast do its earnings and dividends grow, in percent?',
     "parts": ['growth rate (%)'],
     "answer": [5.3999999999999995], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'g = plowback × return on new investment = 0.45 × 12 % = 5.4 %'},
    {"id": 'P1.10', "tier": 1, "group": 'Growth and the value of a business',
     "q": 'A share sells for $36. Earnings next year are $2.70 a share and shareholders want 9 %. What is the present value of its growth opportunities?',
     "parts": ['PVGO ($)'],
     "answer": [5.9999999999999964], "tol": [0.005], "pct": [], "function": '—',
     "working": 'With no growth the share is worth 2.70/0.09 = 30; PVGO = 36 − 30 = 6'},
    {"id": 'P2.1', "tier": 2, "group": 'Bond prices and yields',
     "q": 'An 8-year bond with a face value of $1,000 pays a 5 % coupon once a year. Similar bonds yield 7 %. What is its price?',
     "parts": ['price ($)'],
     "answer": [880.5740298757253], "tol": [0.005], "pct": [], "function": 'PV',
     "working": '50 × 5.9713 + 1,000/1.07⁸ = 298.56 + 582.01 = 880.57: below face value, because the coupon is below the yield'},
    {"id": 'P2.2', "tier": 2, "group": 'Bond prices and yields',
     "q": 'A 6-year bond pays a 6 % coupon in two halves a year. Its yield is quoted as 8 %. What is its price per $1,000 of face value?',
     "parts": ['price per $1,000 of face value ($)'],
     "answer": [906.1492623950163], "tol": [0.005], "pct": [], "function": 'PRICE',
     "working": '12 half-years at 4 %, $30 each: 30 × 9.3851 + 1,000/1.04¹² = 281.55 + 624.60 = 906.15. =PRICE(DATE(2026,1,1), DATE(2032,1,1), 6%, 8%, 100, 2, 0) gives 90.615 per 100'},
    {"id": 'P2.3', "tier": 2, "group": 'Bond prices and yields',
     "q": 'A 5-year bond with a 4 % coupon, paid once a year, sells for $915.75. What is its yield to maturity, in percent?',
     "parts": ['yield to maturity (%)'],
     "answer": [6.00006839500179], "tol": [0.005], "pct": [0], "function": 'RATE',
     "working": 'Try rates: at 5 % the payments are worth 956.71 (too much), at 7 % 876.99 (too little), at 6 % 915.75: the yield is 6 %. =RATE(5, 40, -915.75, 1000, 0)'},
    {"id": 'P2.4', "tier": 2, "group": 'Duration',
     "q": 'A 4-year bond pays a 5 % coupon once a year and yields 6 %. What is its duration, in years?',
     "parts": ['duration (years)'],
     "answer": [3.7177289685302117], "tol": [0.005], "pct": [], "function": 'DURATION',
     "working": 'Present values 47.17 · 44.50 · 41.98 · 831.70 (price 965.35); weights 4.89 % · 4.61 % · 4.35 % · 86.16 %; Σ year × weight = 3.718 years. =DURATION(DATE(2026,1,1), DATE(2030,1,1), 5%, 6%, 1, 0)'},
    {"id": 'P2.5', "tier": 2, "group": 'Duration',
     "q": 'A bond has a duration of 6.3 years at a yield of 5 %. What is its modified duration, and by about what percent does its price change if the yield rises half a point? (Give the change as a negative percent.)',
     "parts": ['modified duration', 'price change (%, negative)'],
     "answer": [6.0, -3.0], "tol": [0.005, 0.005], "pct": [1], "function": 'MDURATION',
     "working": '6.3/1.05 = 6.00; change ≈ −6.00 × 0.5 = −3.00 %'},
    {"id": 'P2.6', "tier": 2, "group": 'The term structure',
     "q": 'Spot rates are 2.5 % for one year, 3 % for two and 3.5 % for three. Price a 3-year bond with a 4 % coupon, paid once a year.',
     "parts": ['price ($)'],
     "answer": [1014.7486405039958], "tol": [0.005], "pct": [], "function": '—',
     "working": '40/1.025 + 40/1.03² + 1,040/1.035³ = 39.02 + 37.70 + 938.02 = 1,014.75'},
    {"id": 'P2.7', "tier": 2, "group": 'Real rates and default risk',
     "q": 'A one-year bond promises $1,070. There is a 6 % chance it defaults and pays only $450. If lenders price it to earn 4.5 % on average, what is its price, and what yield does it promise, in percent?',
     "parts": ['price ($)', 'promised yield (%)'],
     "answer": [988.3253588516747, 8.263942680092939], "tol": [0.005, 0.005], "pct": [1], "function": '—',
     "working": 'Expected payoff 0.94 × 1,070 + 0.06 × 450 = 1,032.80; price 1,032.80/1.045 = 988.33; promised yield 1,070/988.33 − 1 = 8.26 %: the best case, not the expected one'},
    {"id": 'P2.8', "tier": 2, "group": 'Stock prices and dividends',
     "q": 'A share is expected to pay dividends of $1.00, $1.10 and $1.21 over the next three years, and to sell for $26.62 at the end of year three. If shareholders want 12 %, what is it worth today?',
     "parts": ['value of the share ($)'],
     "answer": [21.57861470481049], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": '1.00/1.12 + 1.10/1.12² + (1.21 + 26.62)/1.12³ = 0.89 + 0.88 + 19.81 = 21.58. =NPV(12%, 1, 1.1, 1.21+26.62)'},
    {"id": 'P2.9', "tier": 2, "group": 'Stock prices and dividends',
     "q": 'A share sells for $25 and will pay $1.25 next year. Dividends are expected to grow 4 % a year. What return are shareholders expecting, in percent?',
     "parts": ['expected return (%)'],
     "answer": [9.0], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'r = DIV₁/P₀ + g = 5 % + 4 % = 9 %'},
    {"id": 'P2.10', "tier": 2, "group": 'Growth and the value of a business',
     "q": 'A business will produce free cash flow of $5, $6 and $7 million in the next three years, then grow 2 % a year. At 8 %, what is it worth today ($ millions), and what share of that is the horizon value, in percent?',
     "parts": ['value of the business ($ millions)', "the horizon value's share of it (%)"],
     "answer": [109.79652491998172, 86.03736479842675], "tol": [0.005, 0.005], "pct": [1], "function": 'NPV',
     "working": 'Flows: 15.33. Horizon value at year 3: 7 × 1.02/(0.08 − 0.02) = 119.00, worth 94.47 today. Total 109.80, of which 86.0 % is the horizon value'},
    {"id": 'P3.1', "tier": 3, "group": 'Duration',
     "q": 'Two 10-year bonds both yield 5 %. One pays a 2 % coupon, the other 8 %, once a year. Find each duration, in years. Which price moves more when yields change?',
     "parts": ['duration with the 2 % coupon (years)', 'duration with the 8 % coupon (years)'],
     "answer": [9.014936751442292, 7.541931323711455], "tol": [0.005, 0.005], "pct": [], "function": 'DURATION',
     "working": '2 % coupon: 9.01 years; 8 % coupon: 7.54 years. The low coupon pays more of its value at the end, so its average wait is longer and its price moves more'},
    {"id": 'P3.2', "tier": 3, "group": 'Duration',
     "q": 'A bond with a modified duration of 8.2 fell 2.05 % in price. By about how many percentage points did its yield rise?',
     "parts": ['the rise in the yield (% points)'],
     "answer": [0.25], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'change in yield ≈ price fall / modified duration = 2.05/8.2 = 0.25 of a point'},
    {"id": 'P3.3', "tier": 3, "group": 'The term structure',
     "q": 'A one-year zero-coupon bond sells for $970.87 per $1,000. A two-year bond with a 4 % coupon, paid once a year, sells for $991.19. Find the one-year and the two-year spot rates, in percent.',
     "parts": ['one-year spot rate (%)', 'two-year spot rate (%)'],
     "answer": [3.0, 4.5], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": 'r₁ = 1,000/970.87 − 1 = 3 %. The first coupon is worth 40/1.03 = 38.83, so the market pays 952.36 for 1,040 in year two: (1 + r₂)² = 1,040/952.36 = 1.092025, r₂ = 4.5 %'},
    {"id": 'P3.4', "tier": 3, "group": 'The term structure',
     "q": 'Spot rates are 3 %, 4 % and 5 % for one, two and three years. A 3-year bond with a 5 % coupon is offered at $995. What are its pieces worth, and what does a trader gain per bond by buying it and selling the three payments separately?',
     "parts": ['what the pieces are worth ($)', 'the gain per bond ($)'],
     "answer": [1001.8009784293257, 6.80097842932571], "tol": [0.005, 0.005], "pct": [], "function": '—',
     "working": '50/1.03 + 50/1.04² + 1,050/1.05³ = 1,001.80; the gain is 1,001.80 − 995 = 6.80 a bond, with no risk: the same cash flow at the same date must sell for the same price'},
    {"id": 'P3.5', "tier": 3, "group": 'Real rates and default risk',
     "q": 'An ordinary 10-year bond yields 4.8 %; an indexed bond of the same maturity yields 1.9 % real. At what rate of inflation do the two pay the same, in percent? If inflation turns out to be 3.5 %, what real return does the ordinary bond give, in percent?',
     "parts": ['break-even inflation (%)', "the ordinary bond's real return (%)"],
     "answer": [2.845927379784108, 1.2560386473430052], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": "1.048/1.019 − 1 = 2.85 %. At 3.5 % inflation the ordinary bond's real return is 1.048/1.035 − 1 = 1.26 %, below the indexed bond's 1.9 %"},
    {"id": 'P3.6', "tier": 3, "group": 'Real rates and default risk',
     "q": 'A one-year bond promises $1,060 and sells for $990. In a default it would pay $500. If lenders need 4 % on average, what chance of default does the price build in, in percent?',
     "parts": ['chance of default (%)'],
     "answer": [5.428571428571404], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'Expected payoff must be 990 × 1.04 = 1,029.60. Each point of default chance costs 1,060 − 500 = 560, so p = (1,060 − 1,029.60)/560 = 5.43 %'},
    {"id": 'P3.7', "tier": 3, "group": 'Stock prices and dividends',
     "q": 'A share will pay $3.60 next year and shareholders want 9 %. Value it for growth of 3 %, 6 % and 8 %. Then: it sells for $90. What growth does that price assume, in percent?',
     "parts": ['value at g = 3 % ($)', 'value at g = 6 % ($)', 'value at g = 8 % ($)', 'the growth a $90 price assumes (%)'],
     "answer": [60.00000000000001, 120.00000000000001, 360.00000000000017, 5.0], "tol": [0.005, 0.005, 0.005, 0.005], "pct": [3], "function": '—',
     "working": '60 · 120 · 360: as g → r the value has no limit, so the formula needs g < r. At $90: g = r − DIV₁/P₀ = 9 % − 4 % = 5 %'},
    {"id": 'P3.8', "tier": 3, "group": 'Growth and the value of a business',
     "q": 'A company will earn $6 a share next year and pays out half. Its new investments earn 8 %, and shareholders want 10 %. What is the share worth, and what is the present value of its growth opportunities?',
     "parts": ['value of the share ($)', 'PVGO ($)'],
     "answer": [49.99999999999999, -10.000000000000007], "tol": [0.005, 0.005], "pct": [], "function": '—',
     "working": 'g = 0.5 × 8 % = 4 %; price 3/(0.10 − 0.04) = 50.00. With no growth it would be worth 6/0.10 = 60, so PVGO = -10.00: growth that earns less than shareholders want destroys value'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'With a 4.5 % coupon, what would each $1,000 bond sell for when lenders want 5 %? How much would the whole $80 million issue raise ($ millions)?',
     "parts": ['price of one bond ($)', 'what the issue raises ($ millions)'],
     "answer": [961.3913253540759, 76.91130602832607], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": '45 × 7.7217 + 1,000/1.05¹⁰ = 347.48 + 613.91 = 961.39; 80,000 bonds raise 76.91 million, 3.09 million short'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'What coupon rate (percent) makes the bond sell at exactly its face value? Show the two pieces of its price ($ per bond): the coupons and the final payment.',
     "parts": ['the coupon rate (%)', 'the coupons, valued today ($)', 'the final payment, valued today ($)'],
     "answer": [5.0, 386.0867464592409, 613.9132535407591], "tol": [0.005, 0.005, 0.005], "pct": [0], "function": 'PV',
     "working": 'A bond sells at face value when its coupon rate equals its yield: 5 %. 50 × 7.7217 = 386.09, plus 1,000/1.05¹⁰ = 613.91: 1,000.00'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "The government's ten-year bond yields 4 %. What is the company's credit spread, in percentage points? What would the company's payments be worth if they were certain ($ per bond)? And with 3 % inflation, what real yield do lenders earn, in percent?",
     "parts": ['the credit spread (% points)', 'the payments if they were certain ($)', 'the real yield (%)'],
     "answer": [1.0000000000000002, 1081.1089577935504, 1.9417475728155331], "tol": [0.005, 0.005, 0.005], "pct": [0, 2], "function": 'PV',
     "working": '5 − 4 = 1 point. At 4 %: 50 × 8.1109 + 1,000/1.04¹⁰ = 1,081.11, so the market takes 81.11 a bond off for the risk. Real yield 1.05/1.03 − 1 = 1.94 %'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Find the bond's duration and modified duration at issue. If yields rise to 6 % the next day, estimate the new price with modified duration, then find it exactly ($ per bond).",
     "parts": ['duration (years)', 'modified duration', 'the estimated price at 6 % ($)', 'the exact price at 6 % ($)'],
     "answer": [8.107821675644052, 7.721734929184811, 922.7826507081519, 926.3991294858529], "tol": [0.005, 0.005, 0.005, 0.005], "pct": [], "function": 'DURATION, MDURATION, PRICE',
     "working": 'Duration 8.108 years; modified 8.108/1.05 = 7.722. Estimate: 1,000 × (1 − 0.07722) = 922.78; exactly 926.40 (-7.36 %). The lenders bear that fall; the company still pays 5 %'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Value a share from next year's $0.50 dividend, 5 % growth and a 10 % cost of equity. Then read the cost of equity back from the price, in percent.",
     "parts": ['value of a share ($)', 'the cost of equity read back (%)'],
     "answer": [10.0, 10.0], "tol": [0.005, 0.005], "pct": [1], "function": '—',
     "working": '0.50/(0.10 − 0.05) = 10.00; r = 0.50/10 + 5 % = 10 %'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Where does the 5 % growth come from? Use earnings of $0.80 a share, a payout ratio of 62.5 % and a 13⅓ % return on new investment. Then find the present value of growth opportunities, per share.',
     "parts": ['growth (%)', 'PVGO a share ($)'],
     "answer": [5.0, 2.0], "tol": [0.005, 0.005], "pct": [0], "function": '—',
     "working": 'Plowback 37.5 % × 13.33 % = 5 %. With no growth: 0.80/0.10 = 8.00; PVGO = 10.00 − 8.00 = 2.00. Directly: 0.30 invested earns 0.04 a year, worth 0.40; NPV 0.10, growing 5 %: 0.10/0.05 = 2.00'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'What is a share worth if the new warehouses earn only 10 %? And what if growth were 6 % instead of 5 %?',
     "parts": ['a share if new investment earns 10 % ($)', 'a share at 6 % growth ($)'],
     "answer": [8.0, 12.499999999999998], "tol": [0.005, 0.005], "pct": [], "function": '—',
     "working": 'At 10 % on new investment: g = 37.5 % × 10 % = 3.75 %, price 0.50/0.0625 = 8.00: growth, and no value from it. At 6 % growth: 0.50/0.04 = 12.50: one point of growth is worth a quarter of the price'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Check the share price with a five-year horizon: the present value of five dividends, and of the price in year five. Then give the market value of the shares, and of the company with its bond ($ millions).',
     "parts": ['five dividends, valued today ($)', 'the year-five price, valued today ($)', 'the shares ($ millions)', 'the company ($ millions)'],
     "answer": [2.0752956361649413, 7.924704363835058, 320.0, 400.0], "tol": [0.005, 0.005, 0.005, 0.005], "pct": [], "function": 'NPV',
     "working": 'Dividends 0.50 … 0.6078: 2.08. Price in year five 10 × 1.05⁵ = 12.76, worth 7.92 today: 2.08 + 7.92 = 10.00. Shares 32 × 10 = 320; with the 80 of bonds, 400. Recommend: a 5 % coupon; the shares are worth $10 while new investment keeps earning more than 10 %'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t, i in q["pct"]) for i, (g, w, t) in enumerate(zip(got, q["answer"], q["tol"]))]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't; a rate may be typed as a percentage or as itself
assert right(read_part("70/1.05 + 70/1.05**2 + 1070/1.05**3"), answers("P1.3")[0], 0.005)
assert not right(read_part("1054"), answers("P1.3")[0], 0.005)
assert right(read_part("6.32%"), answers("P1.4")[0], 0.005, True) and right(read_part("0.0632"), answers("P1.4")[0], 0.005, True)
assert not right(read_part("0.0632"), answers("P1.4")[0], 0.005) and not right(read_part("six"), 6.32, 0.005, True)

## Tier 1 · Warm-up

One idea, one step, small numbers: a present value, an annuity factor, a bond's price, a current yield, a
zero-coupon bond, a real rate, a share's expected return, a dividend that grows forever, a growth rate, and the
value of growth opportunities.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Python has no built-in PV or PRICE, and doesn't need one: each formula from the course is a line or two. The cell
below writes them once (the comment beside each names the spreadsheet function that does the same job), then
computes every warm-up answer and checks it against the practice set.

In [ ]:
def pv(amount, r, t): return amount / (1 + r) ** t                 # PV(r, t, 0, -amount, 0)
def annuity_factor(r, t): return 1 / r - 1 / (r * (1 + r) ** t)   # PV(r, t, -1, 0, 0)
def bond_price(face, coupon, y, years, freq=1):                    # -PV(y, years, coupon × face, face, 0), or PRICE
    """An annuity of coupons plus one final payment. With freq coupons a year, the rate for one period is y / freq."""
    n, c, k = years * freq, face * coupon / freq, y / freq
    return c * annuity_factor(k, n) + face / (1 + k) ** n
def real_rate(nominal, inflation): return (1 + nominal) / (1 + inflation) - 1
def expected_return(p0, div1, p1): return (div1 + p1 - p0) / p0    # the dividend and the price gain, over the price
def gordon(div1, r, g): return div1 / (r - g)                      # a dividend growing forever; needs g < r
def pvgo(price, eps1, r): return price - eps1 / r                  # the price = earnings / r + PVGO

p11 = pv(1000, 0.04, 5)                                            # P1.1
p12 = annuity_factor(0.06, 10)                                     # P1.2
p13 = bond_price(1000, 0.07, 0.05, 3)                              # P1.3: 70 a year, then 1,000
p14 = 60 / 950                                                     # P1.4: the current yield
p15 = pv(1000, 0.045, 8)                                           # P1.5: one payment, so its duration is 8 years
p16 = real_rate(0.06, 0.02)                                        # P1.6
p17 = expected_return(40, 1.60, 42.40)                             # P1.7
p18 = gordon(1.50, 0.09, 0.04)                                     # P1.8
p19 = 0.45 * 0.12                                                  # P1.9: plowback × the return on new investment
p110 = pvgo(36, 2.70, 0.09)                                        # P1.10

print(f"PV {p11:,.2f}   annuity factor {p12:.4f}   bond {p13:,.2f}   current yield {p14:.2%}   zero-coupon bond {p15:,.2f}")
print(f"real rate {p16:.4%}   expected return {p17:.0%}   share {p18:,.2f}   growth {p19:.1%}   PVGO {p110:,.2f}")
for k, v in (("P1.1", [p11]), ("P1.2", [p12]), ("P1.3", [p13]), ("P1.4", [100 * p14]), ("P1.5", [p15, 8]),
             ("P1.6", [100 * p16]), ("P1.7", [100 * p17]), ("P1.8", [p18]), ("P1.9", [100 * p19]), ("P1.10", [p110])):
    assert agrees(k, v), k

### The four-year bond, in three lines

The course's bond has a face value of $1,000 and pays a 6 % coupon once a year for four years: $60, $60, $60, then
$1,060. When similar bonds yield 5 %, each payment is divided by 1.05 once for every year it is away, and the
results are added: $1,035.46, the price the video worked out by hand.

In [ ]:
flows = np.array([60, 60, 60, 1060])
years = np.array([1, 2, 3, 4])
price = np.sum(flows / 1.05 ** years)
print(f"price = {price:.2f}")

In [ ]:
# The same bond as an annuity plus one final payment, and at three yields: when yields rise, prices fall.
print("each payment, valued today:", np.round(flows / 1.05 ** years, 2))
print(f"60 × {annuity_factor(0.05, 4):.4f} + 1,000/1.05⁴ = {60 * annuity_factor(0.05, 4):.2f} + {pv(1000, 0.05, 4):.2f} "
      f"= {bond_price(1000, 0.06, 0.05, 4):,.2f}")
for y in (0.05, 0.06, 0.07):
    print(f"at a yield of {y:.0%}: {bond_price(1000, 0.06, y, 4):,.2f}")
assert round(float(price), 2) == round(bond_price(1000, 0.06, 0.05, 4), 2) == 1035.46                # the video's numbers
assert [round(float(v), 2) for v in flows / 1.05 ** years] == [57.14, 54.42, 51.83, 872.06]
assert round(bond_price(1000, 0.06, 0.06, 4), 6) == 1000 and round(bond_price(1000, 0.06, 0.07, 4), 2) == 966.13
assert round(bond_price(1000, 0.06, 0.05, 4, freq=2), 2) == 1035.85 and round(bond_price(100, 0.06, 0.05, 4), 3) == 103.546

### Slide the yield

Two bonds with the same 6 % coupon: one repays in four years (SKY), the other in thirty (CHALK). At a yield of 6 %
both are worth their face value, $1,000. Slide the yield up to 7 %: the four-year bond falls to $966.13, the
thirty-year bond to $875.91. Slide it down and both rise, the long bond much further. The payments never change;
only the yield does, and the longer you wait for your money, the more its price moves.

In [ ]:
def yield_picture(yield_pct=6.0):
    y = yield_pct / 100
    ys = np.linspace(0.005, 0.12, 300)
    short, long_ = bond_price(1000, 0.06, y, 4), bond_price(1000, 0.06, y, 30)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * ys, bond_price(1000, 0.06, ys, 4), color=SKY, label="four years to run")
    ax.plot(100 * ys, bond_price(1000, 0.06, ys, 30), color=CHALK, alpha=0.75, label="thirty years to run")
    ax.axhline(1000, color=DIM, lw=1, ls=":"); ax.text(11.9, 1030, "face value", color=DIM, ha="right", fontsize=11)
    ax.axvline(yield_pct, color=VIOLET, ls="--", lw=1.8)
    ax.scatter([yield_pct, yield_pct], [short, min(long_, 2600)], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 12.3); ax.set_ylim(400, 2600); ax.set_xlabel("yield (% a year)"); ax.set_ylabel("price ($)")
    ax.legend(loc="upper right", frameon=False)
    fig.suptitle(f"yield {yield_pct:g} %: the four-year bond {short:,.2f}, the thirty-year bond {long_:,.2f}",
                 x=0.09, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(bond_price(1000, 0.06, 0.07, 30), 2) == 875.91 and round(bond_price(1000, 0.06, 0.06, 30), 6) == 1000

In [ ]:
interact(yield_picture, yield_pct=FloatSlider(value=6, min=0.5, max=12, step=0.25, description="yield (%)"));

## Tier 2 · Core

The module's standard problems, several steps each: a bond below its face value, coupons twice a year, a yield to
maturity, a duration, a modified duration and the price change it predicts, a bond priced with spot rates, a bond
that may default, a share valued over a three-year horizon, the return a share's price is offering, and a business
valued by its free cash flow.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**A yield has no formula**: a spreadsheet's RATE and YIELD try rates until the payments are worth the price, and
`solve` below does the same by halving an interval. **Duration** is the average wait for your money: each payment's
date, weighted by its share of the price. **Spot rates** discount each payment at the rate for its own date. Each
line at the bottom computes one core answer and checks it against the practice set.

In [ ]:
def solve(f, lo, hi):
    """The x between lo and hi where f(x) = 0, by halving the interval (f must change sign between them)."""
    f_lo = f(lo)
    for _ in range(200):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (f_lo > 0): lo, f_lo = mid, f(mid)
        else: hi = mid
    return (lo + hi) / 2

def ytm(price, face, coupon, years, freq=1):                       # RATE(years, coupon × face, -price, face, 0), or YIELD
    return solve(lambda y: bond_price(face, coupon, y, years, freq) - price, 1e-9, 1)

def duration(face, coupon, y, years, freq=1):                      # DURATION: the average wait, in years
    n, k = years * freq, y / freq
    t = np.arange(1, n + 1)
    pay = np.full(n, face * coupon / freq); pay[-1] += face
    today = pay / (1 + k) ** t
    return float(np.sum(t * today) / np.sum(today) / freq)

def mduration(face, coupon, y, years, freq=1):                     # MDURATION: duration / (1 + the yield per period)
    return duration(face, coupon, y, years, freq) / (1 + y / freq)

def present_value(rate, flows, first=1):                           # NPV(rate, flows): flows[k] arrives at time first + k
    t = first + np.arange(len(flows))
    return float(np.sum(np.asarray(flows, float) / (1 + rate) ** t))

def price_spot(flows, spots):                                      # each payment at the spot rate for its own year
    t = np.arange(1, len(flows) + 1)
    return float(np.sum(np.asarray(flows, float) / (1 + np.asarray(spots, float)) ** t))

p21 = bond_price(1000, 0.05, 0.07, 8)                              # P2.1: the coupon is below the yield
p22 = bond_price(1000, 0.06, 0.08, 6, freq=2)                      # P2.2: 12 half-years at 4 %, $30 each
p23 = ytm(915.75, 1000, 0.04, 5)                                   # P2.3
p24 = duration(1000, 0.05, 0.06, 4)                                # P2.4
modified = 6.3 / 1.05                                              # P2.5
change = -modified * 0.5                                           #        percent, for half a point
p26 = price_spot([40, 40, 1040], [0.025, 0.03, 0.035])             # P2.6
expected = 0.94 * 1070 + 0.06 * 450                                # P2.7: what lenders expect to receive
risky = expected / 1.045
promised = 1070 / risky - 1                                        #        the promised yield: the best case
p28 = present_value(0.12, [1.00, 1.10, 1.21 + 26.62])              # P2.8: three dividends and the price in year three
p29 = 1.25 / 25 + 0.04                                             # P2.9: r = dividend yield + growth
horizon = 7 * 1.02 / (0.08 - 0.02)                                 # P2.10: the value in year 3 of everything after it
business = present_value(0.08, [5, 6, 7]) + pv(horizon, 0.08, 3)
beyond = pv(horizon, 0.08, 3) / business

print(f"bond {p21:,.2f}   half-yearly bond {p22:,.2f}   yield {p23:.2%}   duration {p24:.3f} years   "
      f"modified {modified:.2f}, change {change:.2f} %")
print(f"spot-rate price {p26:,.2f}   risky bond {risky:,.2f}, promising {promised:.2%}   share {p28:.2f}   "
      f"expected return {p29:.0%}")
print(f"business {business:.2f} million, {beyond:.1%} of it beyond the horizon")
for k, v in (("P2.1", [p21]), ("P2.2", [p22]), ("P2.3", [100 * p23]), ("P2.4", [p24]), ("P2.5", [modified, change]),
             ("P2.6", [p26]), ("P2.7", [risky, 100 * promised]), ("P2.8", [p28]), ("P2.9", [100 * p29]),
             ("P2.10", [business, 100 * beyond])):
    assert agrees(k, v), k
assert round(100 * ytm(966.13, 1000, 0.06, 4), 2) == 7.0 and round(100 * ytm(96.613, 100, 0.06, 4), 2) == 7.0   # the video's yield

Duration, payment by payment. The four-year bond at a yield of 5 %: each payment's value today as a share of the
price, and each year multiplied by its share. The shares add up to 1 and the last column adds up to the duration,
3.68 years: the average wait for your money. Divide by 1.05 for the modified duration, 3.50: the percent the price
moves for each point of yield.

In [ ]:
t = np.arange(1, 5)
today = flows / 1.05 ** t
weights = pd.DataFrame({"payment": flows, "value today": today, "share of the price": today / today.sum()},
                       index=pd.Index(t, name="year"))
weights["year × share"] = t * weights["share of the price"]
print(weights.round(4).to_string())
print(f"duration {weights['year × share'].sum():.3f} years   modified duration {weights['year × share'].sum() / 1.05:.3f}")
print(f"a one-point rise in the yield: the estimate {-mduration(1000, 0.06, 0.05, 4):.2f} %, the exact change "
      f"{100 * (bond_price(1000, 0.06, 0.06, 4) / bond_price(1000, 0.06, 0.05, 4) - 1):.2f} %")
assert [round(100 * w, 2) for w in weights["share of the price"]] == [5.52, 5.26, 5.01, 84.22]
assert round(weights["year × share"].sum(), 3) == round(duration(1000, 0.06, 0.05, 4), 3) == 3.679
assert round(mduration(1000, 0.06, 0.05, 4), 3) == 3.504 and round(duration(1000, 0.08, 0.08, 3), 3) == 2.783

### A whole list of bonds, in four lines

An analyst rarely prices one bond. Put the bonds in a table, one row each (the coupon rate, the years to run and
the yield), and the same functions fill in the price, the duration and the modified duration of every one.

In [ ]:
rows = [(0.06, 4, 0.05), (0.04, 10, 0.06), (0.08, 3, 0.08), (0.05, 10, 0.05)]     # coupon rate, years, yield
def each(f, table):                                                # f for every bond in the table, per $1,000 of face value
    return [f(1000, b.coupon, b.yld, b.years) for b in table.itertuples()]

In [ ]:
bonds = pd.DataFrame(rows, columns=["coupon", "years", "yld"])
bonds["price"] = each(bond_price, bonds)
bonds["duration"] = each(duration, bonds)
bonds["modified"] = bonds["duration"] / (1 + bonds["yld"])
print(bonds.round(3).to_string())

In [ ]:
# The video's bonds are in that list: the four-year bond, problem 1's, problem 4's and the case's.
assert [round(v, 2) for v in bonds["price"]] == [1035.46, 852.80, 1000.00, 1000.00]
assert [round(v, 3) for v in bonds["duration"]] == [3.679, round(duration(1000, 0.04, 0.06, 10), 3), 2.783, 8.108]
assert [round(v, 3) for v in bonds["modified"]][::3] == [3.504, 7.722]

### Slide the return on new investment

A share earns $5 next year and pays out 60 % of it, a $3 dividend; shareholders want 10 %. The other 40 % is plowed
back, and growth is the plowback ratio times what the new investment earns. If it earns exactly 10 %, growth is 4 %
and the share is worth $50: the same as with no growth at all. Slide the return up to 15 %: growth is 6 %, the price
is $75, and $25 of it is the present value of growth opportunities (PVGO). Slide it below 10 %: the company still
grows, and the growth takes value away. Growth adds value only when new investment earns more than shareholders want.

In [ ]:
def growth_picture(return_pct=15.0):
    eps, payout, r = 5.0, 0.60, 0.10
    roe = return_pct / 100
    g = (1 - payout) * roe
    price, flat = gordon(eps * payout, r, g), eps / r
    roes = np.linspace(0, 0.215, 300)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * roes, gordon(eps * payout, r, (1 - payout) * roes), color=MINT)
    ax.axhline(flat, color=CHALK, lw=1.4, ls=":")
    ax.text(0.2, flat + 3, "no growth: earnings ÷ r = 50", color=CHALK, fontsize=12)
    ax.axvline(100 * r, color=VIOLET, ls="--", lw=1.8)
    ax.text(100 * r + 0.15, 8, "what shareholders want", color=VIOLET, fontsize=12)
    ax.plot([return_pct, return_pct], [flat, price], color=MINT if price >= flat else CORAL, lw=5, solid_capstyle="butt")
    ax.scatter([return_pct], [price], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 21.5); ax.set_ylim(0, 220); ax.set_xlabel("return on new investment (% a year)")
    ax.set_ylabel("price of a share ($)")
    sign = "+" if price >= flat else "−"
    fig.suptitle(f"new investment earns {return_pct:g} %: growth {100 * g:.1f} %, price {price:,.2f} = {flat:.2f} "
                 f"{sign} PVGO {abs(price - flat):,.2f}", x=0.08, ha="left", fontsize=15,
                 color=GOLD if price >= flat else CORAL)
    footnote(fig); plt.show()

assert [round(gordon(3, 0.10, 0.4 * roe), 2) for roe in (0.10, 0.15, 0.08)] == [50, 75, 44.12]

In [ ]:
interact(growth_picture, return_pct=FloatSlider(value=15, min=4, max=20, step=0.5, description="return (%)"));

## Tier 3 · Stretch

Combine ideas, spot the trap, run a formula backwards: two bonds whose coupons change their durations, a yield
change read from a price fall, spot rates taken out of two prices, a bond that sells for less than its pieces,
break-even inflation, the chance of default a price builds in, what happens as g approaches r, and growth that
destroys value.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Running a formula backwards.** A zero-coupon bond's price gives its spot rate directly; a coupon bond's price,
less the value of its first coupon, gives the next one. A risky bond's price, carried forward at the rate lenders
need, is the payoff they expect, and that gives the chance of default. Each line computes one stretch answer and
checks it against the practice set.

In [ ]:
low, high = duration(1000, 0.02, 0.05, 10), duration(1000, 0.08, 0.05, 10)   # P3.1: the low coupon waits longer
yield_rise = 2.05 / 8.2                                            # P3.2: the price fall over the modified duration
zero, two_year = 970.87, 991.19                                    # P3.3: the two prices, to the cent
r1 = 1000 / zero - 1                                               #        one payment, so the one-year rate directly
left = two_year - 40 / (1 + r1)                                    #        what the market pays for the 1,040 in year two
r2 = (1040 / left) ** 0.5 - 1
pieces = price_spot([50, 50, 1050], [0.03, 0.04, 0.05])            # P3.4: the three payments sold separately
gain = pieces - 995
break_even = real_rate(0.048, 0.019)                               # P3.5: the inflation at which the two bonds pay the same
real_return = real_rate(0.048, 0.035)
chance = (1060 - 990 * 1.04) / (1060 - 500)                        # P3.6: each point of default chance costs 560
values = [gordon(3.60, 0.09, g) for g in (0.03, 0.06, 0.08)]       # P3.7
implied_g = 0.09 - 3.60 / 90                                       #        g = r − dividend / price
poor = gordon(6 * 0.5, 0.10, 0.5 * 0.08)                           # P3.8: growth 4 %, from investments that earn 8 %
poor_pvgo = pvgo(poor, 6, 0.10)

print(f"durations {low:.2f} and {high:.2f} years   yield rise {yield_rise:.2f} of a point   "
      f"spot rates {r1:.2%} and {r2:.2%}")
print(f"the pieces {pieces:,.2f}, a gain of {gain:.2f} a bond   break-even inflation {break_even:.2%}, "
      f"real return {real_return:.2%}   chance of default {chance:.2%}")
print("share at g = 3 %, 6 %, 8 %:", ", ".join(f"{v:,.2f}" for v in values), f"  the growth a $90 price assumes {implied_g:.0%}")
print(f"share {poor:.2f}, PVGO {poor_pvgo:.2f}: growth that earns less than shareholders want takes value away")
for k, v in (("P3.1", [low, high]), ("P3.2", [yield_rise]), ("P3.3", [100 * r1, 100 * r2]), ("P3.4", [pieces, gain]),
             ("P3.5", [100 * break_even, 100 * real_return]), ("P3.6", [100 * chance]),
             ("P3.7", values + [100 * implied_g]), ("P3.8", [poor, poor_pvgo])):
    assert agrees(k, v), k
assert low > high and poor_pvgo < 0

## Tier 4 · The case: Bangalore Cold Chain's bond and its shares

Bangalore Cold Chain runs refrigerated warehouses in four countries (a fictional company; our own data). It
replaces its bank loans with one ten-year bond, while the market prices its shares:

| | |
|---|---|
| the bond | $80 million: 80,000 bonds of $1,000, ten years, one coupon a year |
| yields | the government's ten-year bond yields 4 %; lenders want 5 % from the company; expected inflation is 3 % |
| the shares | 32 million shares; next year's dividend $0.50 a share, from earnings of $0.80 |
| growth | the dividend is expected to grow 5 % a year; shareholders want 10 % |

**What coupon must the bond carry, how far can its price move, what is a share worth, and how much of that price is
growth?** The next cell builds the bond's ten dated payments and the share's first ten dividends as two DataFrames
(the same tables as the spreadsheet's Case study tab), and checks a few of their numbers.

In [ ]:
ISSUE_M, FACE, YEARS = 80, 1000, 10                               # $80 million in bonds of $1,000, ten years to run
COUPON_LOW, COUPON = 0.045, 0.05                                  # the coupon first proposed · the coupon the bond carries
GOVERNMENT, LENDERS_WANT, INFLATION = 0.04, 0.05, 0.03
SHARES_M, DIVIDEND, EARNINGS = 32, 0.50, 0.80                     # millions of shares · next year's dividend and earnings
GROWTH, COST_OF_EQUITY, RETURN_ON_NEW = 0.05, 0.10, 0.05 / 0.375  # the return on new investment is 13⅓ %

year = np.arange(1, YEARS + 1)
bond = pd.DataFrame({"year": year, "date": [pd.Timestamp(2027 + int(k), 1, 1) for k in year]})
bond["payment"] = COUPON * FACE + np.where(year == YEARS, FACE, 0.0)
for rate in (0.04, 0.05, 0.06):
    bond[f"factor at {rate:.0%}"] = 1 / (1 + rate) ** year         # the discount factor for each date
dividends = pd.DataFrame({"year": year, "date": bond["date"]})
dividends["dividend"] = DIVIDEND * (1 + GROWTH) ** (year - 1)
dividends["factor"] = 1 / (1 + COST_OF_EQUITY) ** year
dividends["price at year end"] = gordon(DIVIDEND, COST_OF_EQUITY, GROWTH) * (1 + GROWTH) ** year

# The case's numbers as the course computed them, to check these tables against.
# @STATS
CASE_STATS = {'bonds': 80000.0,
 'low_price': 961.3913253540759,
 'low_raised_m': 76.91130602832607,
 'low_short_m': 3.0886939716739334,
 'coupon': 0.05,
 'price': 1000.0,
 'factor': 7.721734929184818,
 'coupons_pv': 386.0867464592409,
 'face_pv': 613.9132535407591,
 'interest_m': 4.0,
 'spread': 0.010000000000000002,
 'safe_price': 1081.1089577935504,
 'risk_cut': 81.10895779355042,
 'real': 0.01941747572815533,
 'dur': 8.107821675644052,
 'mdur': 7.721734929184811,
 'at6': 926.3991294858529,
 'drop6': -0.0736008705141471,
 'est6': -0.07721734929184812,
 'wait_cost_m': 0.8,
 'share': 10.0,
 'yield': 0.05,
 'r_back': 0.1,
 'plow': 0.375,
 'kept': 0.30000000000000004,
 'g_from': 0.05,
 'nogrowth': 8.0,
 'pvgo': 2.0,
 'extra': 0.04000000000000001,
 'npv1': 0.10000000000000003,
 'pvgo_direct': 2.0000000000000004,
 'g_flat': 0.037500000000000006,
 'share_flat': 8.0,
 'share_g6': 12.499999999999998,
 'divs5': [0.5, 0.525, 0.55125, 0.5788125000000001, 0.6077531250000001],
 'p5': 12.762815625000004,
 'divs5_pv': 2.0752956361649413,
 'p5_pv': 7.924704363835058,
 'horizon_share': 0.7924704363835058,
 'equity_m': 320.0,
 'value_m': 400.0,
 'debt_share': 0.2,
 'pvgo_m': 64.0}
# @END STATS
S = CASE_STATS
assert len(bond) == 10 and bond["payment"].sum() == 1500 and bond["payment"].iloc[-1] == 1050
assert abs((bond["payment"] * bond["factor at 5%"]).sum() - S["price"]) < 1e-9
assert all(abs(a - b) < 1e-12 for a, b in zip(dividends["dividend"], S["divs5"]))
print(f"{len(bond)} payments on the bond, {len(dividends)} dividends: the same cash flows as the course's case.")
bond.head(3)

### How you'd do this in Python

With the payments in a DataFrame, a price is one column times a column of discount factors, summed, and the
duration is each year weighted by its share of that price. The formulas from the course give the same numbers in
one line each. The cell below reproduces the video's case: the coupon that falls short, the coupon that sells at
face value, the price at three yields, the duration, then the share and how much of it is growth.

In [ ]:
million = 1e6
low_price = bond_price(FACE, COUPON_LOW, LENDERS_WANT, YEARS)                             # a 4.5 % coupon when lenders want 5 %
raised = low_price * ISSUE_M / FACE                                                       # $ millions, for 80,000 bonds
coupons_today = COUPON * FACE * annuity_factor(LENDERS_WANT, YEARS)                       # the 5 % coupon: an annuity…
face_today = pv(FACE, LENDERS_WANT, YEARS)                                                # …plus one final payment
priced = {rate: (bond["payment"] * bond[f"factor at {rate:.0%}"]).sum() for rate in (0.04, 0.05, 0.06)}
share_of_price = bond["payment"] * bond["factor at 5%"] / priced[0.05]
wait = (bond["year"] * share_of_price).sum()                                              # duration: the average wait
modified_wait = wait / (1 + LENDERS_WANT)
share = gordon(DIVIDEND, COST_OF_EQUITY, GROWTH)
no_growth = EARNINGS / COST_OF_EQUITY
growth_value = pvgo(share, EARNINGS, COST_OF_EQUITY)
today_div = dividends["dividend"] * dividends["factor"]                                   # each dividend, valued today
today_price = dividends["price at year end"] * dividends["factor"]                        # each year-end price, valued today

print(f"1 · a {COUPON_LOW:.1%} coupon: {low_price:,.2f} a bond, so the issue raises {raised:.2f} million, "
      f"{ISSUE_M - raised:.2f} short")
print(f"2 · a {COUPON:.0%} coupon: {coupons_today:.2f} + {face_today:.2f} = {coupons_today + face_today:,.2f}: face value")
print("3 · the same payments at 4 %, 5 %, 6 %: " + "   ".join(f"{v:,.2f}" for v in priced.values())
      + f"   (the real yield at 5 %: {real_rate(LENDERS_WANT, INFLATION):.2%})")
print(f"4 · duration {wait:.3f} years, modified {modified_wait:.3f}: a one-point rise is estimated at "
      f"{-modified_wait:.2f} %, and is exactly {100 * (priced[0.06] / priced[0.05] - 1):.2f} %")
print(f"5 · a share: {DIVIDEND:.2f}/({COST_OF_EQUITY:.2f} − {GROWTH:.2f}) = {share:.2f} = {no_growth:.2f} with no "
      f"growth + PVGO {growth_value:.2f}")
print(f"6 · five years of dividends {today_div[:5].sum():.2f} + the year-five price {today_price[4]:.2f} = "
      f"{today_div[:5].sum() + today_price[4]:.2f}; the shares {SHARES_M * share:.0f} million, the company "
      f"{SHARES_M * share + ISSUE_M:.0f} million")
assert abs(low_price - S["low_price"]) < 1e-9 and abs(raised - S["low_raised_m"]) < 1e-9
assert abs(coupons_today - S["coupons_pv"]) < 1e-9 and abs(face_today - S["face_pv"]) < 1e-9
assert abs(priced[0.04] - S["safe_price"]) < 1e-9 and abs(priced[0.06] - S["at6"]) < 1e-9
assert abs(wait - S["dur"]) < 1e-9 and abs(modified_wait - S["mdur"]) < 1e-9
assert abs(wait - duration(FACE, COUPON, LENDERS_WANT, YEARS)) < 1e-9
assert abs(share - S["share"]) < 1e-9 and abs(growth_value - S["pvgo"]) < 1e-9 and abs(no_growth - S["nogrowth"]) < 1e-9
assert abs(today_div[:5].sum() - S["divs5_pv"]) < 1e-9 and abs(today_price[4] - S["p5_pv"]) < 1e-9
assert [round(v, 2) for v in (low_price, raised, priced[0.04], priced[0.06], wait, modified_wait)] == \
       [961.39, 76.91, 1081.11, 926.40, 8.11, 7.72]                                                   # the video's numbers
assert round(low_price / 10, 3) == 96.139 and round(wait, 3) == 8.108 and round(modified_wait, 3) == 7.722

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [3, 2]})
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14, wspace=0.28)
bars = ax1.bar(["4 %\nif the payments\nwere certain", "5 %\nwhat lenders want", "6 %\nif yields rise\na point"],
               list(priced.values()), width=0.55, color=SKY)
bars[1].set_color(GOLD)
for b, v in zip(bars, priced.values()):
    ax1.text(b.get_x() + b.get_width() / 2, v + 12, f"{v:,.2f}", ha="center", color=CHALK)
ax1.axhline(FACE, color=DIM, lw=1, ls=":"); ax1.set_ylim(0, 1250); ax1.set_ylabel("price of one bond ($)")
ax1.tick_params(axis="x", labelsize=10)
ax2.bar(["a share"], [no_growth], width=0.5, color=CHALK, alpha=0.6, label="earnings ÷ r: no growth")
ax2.bar(["a share"], [growth_value], bottom=[no_growth], width=0.5, color=MINT, label="PVGO: growth to come")
ax2.text(0, no_growth / 2, f"{no_growth:.2f}", ha="center", va="center", color=SLATE, fontsize=14)
ax2.text(0, no_growth + growth_value / 2, f"{growth_value:.2f}", ha="center", va="center", color=SLATE, fontsize=14)
ax2.set_xlim(-0.9, 0.9); ax2.set_ylim(0, 13.5); ax2.set_ylabel("$ a share"); ax2.legend(loc="upper center", frameon=False)
fig.suptitle(f"A {100 * COUPON:g} % coupon sells the bond at face value; a share is worth {share:.2f}, "
             f"{growth_value:.2f} of it growth", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the bond's price if yields rise to 6 %, the share if the new
warehouses earn only 10 % or if growth were 6 %, a five-year check on the share price, and your own recommendation
with the numbers that anchor it. Use the DataFrames `bond` and `dividends` and the functions above in a cell of your
own (press **+** to add one), then type your answers: dollars to the cent, rates in percent, totals in $ millions.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrames

Try the questions first. This cell works every tier 4 answer out from `bond`, `dividends` and the functions above
and checks it against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
plowback = 1 - DIVIDEND / EARNINGS                                                        # 37.5 % of earnings kept
mine = {}
mine["P4.1"] = [low_price, raised]
mine["P4.2"] = [100 * LENDERS_WANT, coupons_today, face_today]                            # the coupon rate = the yield
mine["P4.3"] = [100 * (LENDERS_WANT - GOVERNMENT), priced[0.04], 100 * real_rate(LENDERS_WANT, INFLATION)]
mine["P4.4"] = [wait, modified_wait, FACE * (1 - modified_wait * 0.01), priced[0.06]]
mine["P4.5"] = [share, 100 * (DIVIDEND / share + GROWTH)]
mine["P4.6"] = [100 * plowback * RETURN_ON_NEW, growth_value]
mine["P4.7"] = [gordon(DIVIDEND, COST_OF_EQUITY, plowback * COST_OF_EQUITY), gordon(DIVIDEND, COST_OF_EQUITY, 0.06)]
mine["P4.8"] = [today_div[:5].sum(), today_price[4], SHARES_M * share, SHARES_M * share + ISSUE_M]

for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 4))}" for p, v in zip(Q[k]["parts"], got)))
assert abs(mine["P4.7"][0] - S["share_flat"]) < 1e-9 and abs(mine["P4.7"][1] - S["share_g6"]) < 1e-9
print("\nEvery tier 4 answer worked out from the DataFrames matches the practice set.")
print(f"The bond needs a {COUPON:.0%} coupon to raise the full {ISSUE_M} million; a one-point rise in yields would take "
      f"about {-100 * (priced[0.06] / priced[0.05] - 1):.1f} % off its price. A share is worth {share:.2f}, of which "
      f"{growth_value:.2f} is growth that only holds while new investment earns more than {COST_OF_EQUITY:.0%}.")

## Sources
Every exercise, example and company here is our own (Bangalore Cold Chain is fictional). The order of topics follows
the bond and share valuation chapters of a standard corporate finance course; the formulas are the standard ones
for a bond's price and yield to maturity, spot rates, real rates, the dividend discount model with constant growth,
and the present value of growth opportunities. Duration is Frederick Macaulay's measure (Some Theoretical Problems
Suggested by the Movements of Interest Rates, Bond Yields and Stock Prices in the United States since 1856, National
Bureau of Economic Research, 1938). The spreadsheet functions named in the hints (PV, RATE, PRICE, YIELD, DURATION,
MDURATION, DATE, NPV) have the same names and take their arguments in the same order in the two most widely used
spreadsheets (their official function lists, retrieved 2026-11-11).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/cf02_bonds_stocks/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("cf02_practice", _src.resolve())
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        _t = _e["tol"] if isinstance(_e["tol"], tuple) else (_e["tol"],) * len(_q["answer"])
        assert list(_t) == _q["tol"], _e["id"]
    assert len(_P.EX) == len(EX) and set(_P.CS) == set(CASE_STATS)
    assert all(abs(a - b) < 1e-9 for k, v in CASE_STATS.items() for a, b in zip(_flat(v), _flat(_P.CS[k])))
    _C = _P.CASE
    assert (_C["face_m"], _C["bond_face"], _C["years"]) == (ISSUE_M, FACE, YEARS)
    assert (_C["coupon_low"], _P.CS["coupon"], _C["gov"], _C["y"], _C["inflation"]) == \
           (COUPON_LOW, COUPON, GOVERNMENT, LENDERS_WANT, INFLATION)
    assert (_C["shares_m"], _C["div1"], _C["eps1"], _C["g"], _C["r"]) == (SHARES_M, DIVIDEND, EARNINGS, GROWTH, COST_OF_EQUITY)
    assert abs(_C["roe_new"] - RETURN_ON_NEW) < 1e-12 and _P.GC.endswith("Bangalore Cold Chain")
    assert rows[0] == (_P.BOND["coupon"], _P.BOND["years"], _P.BOND["y"]) and rows[1] == (_P.Q1["coupon"], _P.Q1["years"], _P.Q1["y"])
    assert rows[2] == (_P.Q4["coupon"], _P.Q4["years"], _P.Q4["y"]) and rows[3] == (COUPON, YEARS, LENDERS_WANT)
    assert abs(bonds["duration"][0] - _P.DUR) < 1e-9 and abs(bonds["modified"][0] - _P.MDUR) < 1e-9
    assert abs(ytm(_P.YTM_PRICE, 1000, 0.06, 4) - _P.YTM) < 1e-9 and abs(price_spot(_P.BOND_FLOWS, _P.SPOTS) - _P.SPOT_PRICE) < 1e-9
    print("matches the course's practice set")